# 02. Causal Conv3D, размеры и пределы причинности

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 4](https://diffusion.csail.mit.edu/2026/docs/20260128_Lecture_04_edited.pdf).
<!-- study-navigation:end -->

Из непрерывной идеи зависимости только от прошлого получим дискретную свёртку, padding, down/up sampling и проверим строгую причинность каждого оператора.

[Порядок и источники](../README.md) · [Связь с MIT Lab 3](../MIT_LAB3_BRIDGE.md) · [Все компоненты и методы](../SOURCE_MAP.md) · [Результаты](../reports/VALIDATION.md).

Работаем по циклу **формула → прогноз → код → измерение → вывод**. Все ячейки по умолчанию используют CPU; никакие веса, датасеты или ComfyUI jobs не скачиваются/запускаются. Исходные алгоритмы и учебные адаптеры явно различаются. Перед запуском численного опыта сформулируйте ожидаемый результат.


**MIT мост:** Lab 3 Q4.1 ResidualBlock (56–57), Q4.3 EncoderBlock (60–61), Q4.5 DecoderBlock (64–65). В MIT — 2D MNIST; здесь добавлены time axis, causal padding и first-frame rule. Tiling/streaming — сверх MIT. [Точные соответствия](../MIT_LAB3_BRIDGE.md).


In [ ]:
from pathlib import Path
import sys, json, math
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
candidates=[Path.cwd(),*Path.cwd().parents]
COURSE=next((p/'vae' for p in candidates if (p/'vae/labkit').is_dir()),None)
assert COURSE is not None, 'Clone the repository first; see the Colab instructions in the course README. No weights are needed.'
sys.path.insert(0,str(COURSE))
from labkit.vae_lab import (NATIVE,TinyCodec,DiagonalPosterior,oracle_codec,plan,
    axis_coverage,anchored_starts,synthetic_video,errors,measure,show_source,
    source_hash,save_metrics,figure_path)
torch.set_num_threads(2)
torch.manual_seed(42)
LAB='02_causal_convolution_and_geometry';metrics={}
print('Pinned source:',source_hash())


## 1. Дискретная причинная свёртка

Для фиксированных каналов и spatial coordinates causal filter использует x[t], x[t−1], …:

$$
y_{b,o,t,h,w}=b_o+\sum_{c,i,j,k}W_{o,c,i,j,k}\,
 x_{b,c,\,s_t t+i-p_t,\,s_h h+j-p_h,\,s_w w+k-p_w}.
$$

При temporal kernel K и dilation d нужна левая добавка pₜ=d(K−1), правая 0. Тогда самый поздний input index равен sₜt. Spatial padding симметричен. `replicate` повторяет первый кадр, а не создаёт физически существовавшее прошлое.

Размер одной оси (Pₗ/Pᵣ — F.pad, p — внутренний padding Conv3d):

$$
L_{out}=\left\lfloor\frac{L+P_l+P_r+2p-d(K-1)-1}{s}\right\rfloor+1.
$$

Для K=3, d=1, temporal Pₗ=2, Pᵣ=0 получаем ceil(L/s). Два temporal stride=2 дают floor((F−1)/4)+1. **Не подменяйте Conv3d tuple (T,H,W) порядком F.pad (W,H,T)**.


In [ ]:
show_source('HunyuanVideoCausalConv3d')
conv=NATIVE.HunyuanVideoCausalConv3d(1,1,3,bias=False)
x=torch.randn(1,1,9,8,8);changed=x.clone();changed[:,:,5:]+=7
with torch.inference_mode():
    delta=(conv(x)[:,:,:5]-conv(changed)[:,:,:5]).abs().max()
assert delta==0
metrics['causal_conv_prefix_error']=float(delta)
for n in [1,2,5,9,10]:
    down=NATIVE.HunyuanVideoCausalConv3d(1,1,3,stride=(2,1,1))
    assert down(torch.zeros(1,1,n,8,8)).shape[2]==math.ceil(n/2)


## 2. Проверяем условия реализации, а не имя класса

В приложенной версии padding построен из kernel tuple в порядке, который совпадает с нужным только для используемых изотропных kernels (1 или 3). Также dilation не входит в вычисление causal padding. Это ограничение general constructor, не доказательство поломки released architecture: в ней надо отдельно проверить фактические call sites.

Предскажите output shape для kernel=(3,3,5) и dilation=2. В корректном same-spatial/causal-time варианте H/W не меняются и future influence исключён. Ниже лишь конструкторный edge case; типовой k=3,d=1 уже прошёл.


In [ ]:
x=torch.zeros(1,1,9,12,12)
for kwargs in [{'kernel_size':(3,3,5)},{'kernel_size':3,'dilation':2}]:
    c=NATIVE.HunyuanVideoCausalConv3d(1,1,**kwargs)
    print(kwargs,'padding=',c.time_causal_padding,'output=',tuple(c(x).shape))
metrics['constructor_limits']='anisotropic kernel and dilation require a separate padding audit; default isotropic path tested'


## 3. Почему special first frame меняет inverse geometry

Spatial interpolation удваивает каждый spatial index. Temporal interpolation удваивает только F−1 кадров после первого:

$$
U_r(F)=1+r(F-1),\qquad U_2(U_2(F))=1+4(F-1).
$$

Обычное F→2F дважды дало бы 4F и лишние три кадра. Native `HunyuanVideoUpsampleCausal3D.forward` отдельно обрабатывает first_frame. Проверим F=1 и F>1 на оригинальном классе.


In [ ]:
show_source('HunyuanVideoUpsampleCausal3D')
up=NATIVE.HunyuanVideoUpsampleCausal3D(1,upsample_factor=(2,2,2))
for n in [1,2,5]:
    y=up(torch.zeros(1,1,n,4,4));assert y.shape==(1,1,2*n-1,8,8)
    print(n,'->',tuple(y.shape))


## 4. Receptive field и GroupNorm: где теряется локальность

Для последовательных convolution layers с jump J и receptive field R:

$$
J_0=1,\quad R_0=1,\qquad
R_l=R_{l-1}+(K_l-1)d_lJ_{l-1},\quad J_l=s_lJ_{l-1}.
$$

Это размер **convolutional path**. Его нельзя считать конечным receptive field всего VAE с глобальной нормализацией. GroupNorm на B,C,F,H,W вычисляет group mean/variance по channels group и всем F/H/W:

$$
\mu_g=\frac{1}{m}\sum_{i\in g}x_i,\qquad
\hat x_i=(x_i-\mu_g)/\sqrt{v_g+\varepsilon}.
$$

Изменение будущего кадра меняет μ и v для прошлого. `eval()` не замораживает эти статистики, как BatchNorm. Значит **causal conv ≠ strict online causality всей сети**. Аналогично spatial tile меняет статистики. Causal attention mask отдельно запрещает future attention, но не отменяет утечку через normalization.


In [ ]:
x=torch.randn(1,4,9,8,8);changed=x.clone();changed[:,:,5:]+=5
norm=nn.GroupNorm(2,4).eval()
with torch.inference_mode():delta=float((norm(x)[:,:,:5]-norm(changed)[:,:,:5]).abs().max())
assert delta>1e-3
metrics['groupnorm_future_prefix_error']=delta
mask=NATIVE.prepare_causal_attention_mask(3,2,torch.float32,torch.device('cpu'),1)[0]
assert torch.isneginf(mask[0,2:]).all() and torch.isfinite(mask[2,:4]).all()
fig,ax=plt.subplots(figsize=(4,3));ax.imshow(torch.isfinite(mask),cmap='Blues');ax.set(xlabel='Key token',ylabel='Query token',title='3 frames × 2 spatial tokens: allowed attention')
fig.tight_layout();fig.savefig(figure_path(LAB,'causal_mask'));plt.show()
J=R=1
for k,s in [(3,1),(3,2),(3,2)]:R+=(k-1)*J;J*=s
metrics['example_conv_path']={'receptive_field':R,'jump':J}
print('GroupNorm prefix change:',delta,'convolution path RF:',R)


## Упражнения и критерий перехода

1. Повторите future perturbation для native ResnetBlock, затем для блока без GroupNorm.
2. Найдите halo для трёх k=3 conv со stride=1. Почему с GroupNorm этого halo недостаточно?
3. Докажите temporal output formula с first-frame rule. Переход: различайте causal convolution, attention mask и strict streaming contract.

Сначала запишите прогноз, затем меняйте один фактор. Разделяйте ошибку аппроксимации, ошибку интерфейса/формы и аппаратное ограничение. Сохранённые метрики относятся к этому запуску.


In [ ]:
result=save_metrics(LAB,metrics)
print(json.dumps(result,ensure_ascii=False,indent=2))


## Источники

[HunyuanVideo, v6, §4.1 / Fig. 6 / Eq. 1, стр. 5–6](https://arxiv.org/pdf/2412.03603v6) · [Auto-Encoding Variational Bayes](https://arxiv.org/abs/1312.6114) · [Официальный HunyuanVideo](https://github.com/Tencent-Hunyuan/HunyuanVideo) · [Закреплённый исследуемый исходник](../../kandinsky-6/reference/kandinsky-6/kandinsky/core/components/hunyuan_vae.py) · [Lecture 4 / исходный MIT Lab 3](../../lecture_04/README.md).

Математические выводы и малые опыты — авторские пояснения. Числа tiny-модели не являются оценкой pretrained HunyuanVideo/Kandinsky. Подробная provenance и ограничения — в SOURCE_MAP.
